# File-Based Observability with the NeMo Agent Toolkit

This notebook shows the smallest useful observability loop in NAT: configure a local file trace exporter, run one simple chat workflow, and inspect the trace records that were written to disk.

The point is not to introduce another hosted service yet. A local file trace makes the instrumentation concrete first: you can see the workflow span, the LLM call, timing, input/output payloads, and errors without needing LangSmith, Phoenix, Langfuse, or an OpenTelemetry collector.

## 0. Setup

Use the same environment as the earlier notebooks. If `requirements.txt` is already installed into this kernel, you can skip the install cell.

In [1]:
# Only needed if you did not install requirements.txt into this kernel's environment already.
# %pip install -r ../requirements.txt

In [2]:
import os
from getpass import getpass
from pathlib import Path

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")

# Leave unset for the public build.nvidia.com endpoint, or set this for an internal/self-hosted NIM gateway.
os.environ.setdefault("NVIDIA_BASE_URL", "")

'https://inference-api.nvidia.com/v1/'

## 1. Configure Local File Tracing

NAT observability is configured under `general.telemetry.tracing`. The `file` exporter is built in, so this example does not need a vendor account or extra service.

`mode: overwrite` keeps each run easy to inspect in a notebook. For longer-running demos, switch it to `append` or enable rolling files.

In [3]:
%%writefile observability_config.yml
general:
  telemetry:
    tracing:
      local_file:
        _type: file
        project: nv-agent-toolkit-file-tracing
        output_path: ./traces/nat-file-trace.jsonl
        mode: overwrite

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nvidia/Nemotron-3-Nano-30B-A3B
    max_tokens: 300
    temperature: 0.0
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}
    thinking: false

workflow:
  _type: chat_completion
  name: traced_chat_completion
  llm_name: nim_llm
  system_prompt: >
    You are a concise assistant. Answer in two short sentences or fewer.

Writing observability_config.yml


## 2. Sanity-Check the Config

Before making a model call, load the config locally. This catches misspelled fields and confirms that tracing is attached to the workflow config.

In [4]:
from nat.runtime.loader import load_config

config = load_config("observability_config.yml")
print("workflow type:", config.workflow.type)
print("workflow name:", config.workflow.name)
print("tracing exporters:", list(config.general.telemetry.tracing.keys()))
print(config.general.telemetry.tracing["local_file"])

workflow type: chat_completion
workflow name: traced_chat_completion
tracing exporters: ['local_file']
output_path='./traces/nat-file-trace.jsonl' project='nv-agent-toolkit-file-tracing' mode=<FileMode.OVERWRITE: 'overwrite'> enable_rolling=False max_file_size=10485760 max_files=5 cleanup_on_init=False


## 3. Run a Traced Workflow

This is the only cell that calls the hosted model. When it completes, NAT should write a local trace file at `traces/nat-file-trace.jsonl`.

In [5]:
from nat.utils import run_workflow

result = await run_workflow(
    config_file="observability_config.yml",
    prompt="In one short answer, what does observability show us in an agent workflow?",
)
print(result)


Observability reveals the internal state, performance metrics, and decision pathways of an agent workflow, letting us monitor, debug, and optimize its behavior in real time.


/Users/siddharthab/Documents/Code/personal/nv-agent-toolkit/.venv/lib/python3.12/site-packages/nat/builder/function.py:380: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  return await self._ainvoke_fn(value)


## 4. Locate the Trace File

A missing or empty file usually means the workflow did not finish, the config was not the one being run, or the process exited before telemetry flushed.

In [6]:
trace_path = Path("traces/nat-file-trace.jsonl")
print("trace path:", trace_path.resolve())
print("exists:", trace_path.exists())
print("bytes:", trace_path.stat().st_size if trace_path.exists() else 0)

trace path: /Users/siddharthab/Documents/Code/personal/nv-agent-toolkit/notebooks/traces/nat-file-trace.jsonl
exists: True
bytes: 1422


## 5. Inspect the Trace Records

The exact schema can evolve, so this cell is intentionally defensive. It parses JSON lines when possible, shows the top-level keys, and then gives a raw preview so you can still inspect the file if the exporter writes a slightly different JSON shape.

In [9]:
import json

records = []
if trace_path.exists():
    for line in trace_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

print(f"parsed JSON records: {len(records)}")
for i, record in enumerate(records[:5], start=1):
    print(f"\nrecord {i} keys:", sorted(record.keys()))
    name = record.get("name") or record.get("span_name") or record.get("operation_name")
    if name:
        print("name:", name)
    start = record.get("start_time") or record.get("start_time_unix_nano")
    end = record.get("end_time") or record.get("end_time_unix_nano")
    if start or end:
        print("time:", start, "->", end)

print("\nraw preview:")
print(trace_path.read_text(encoding="utf-8")[:2000] if trace_path.exists() else "Run the workflow cell first.")

parsed JSON records: 4

record 1 keys: ['function_ancestry', 'parent_id', 'payload']

record 2 keys: ['function_ancestry', 'parent_id', 'payload']

record 3 keys: ['function_ancestry', 'parent_id', 'payload']

record 4 keys: ['function_ancestry', 'parent_id', 'payload']

raw preview:
{"parent_id":"root","function_ancestry":{"function_id":"root","function_name":"root","parent_id":null,"parent_name":null},"payload":{"event_type":"WORKFLOW_START","event_timestamp":1783006535.193402,"span_event_timestamp":null,"framework":null,"name":"traced_chat_completion","tags":null,"metadata":{"chat_responses":null,"chat_inputs":null,"tool_inputs":null,"tool_outputs":null,"tool_info":null,"span_inputs":null,"span_outputs":null,"provided_metadata":{"workflow_run_id":"d812b0d5-6d2b-4889-9b51-b76e6b5676dc","workflow_trace_id":"ddcc2333d1124635942dba9977754979","conversation_id":null,"display_name":"traced_chat_completion"},"tools_schema":[]},"data":{"input":"In one short answer, what does observability s

## 6. What to Look For

Useful trace evidence usually answers these questions:

- Did the workflow start and finish successfully?
- Which LLM was called, with which prompt and model settings?
- How long did the workflow and LLM span take?
- Was there an exception, retry, parsing failure, or tool call?
- Did the final response match what the workflow was supposed to produce?

For vendor-backed observability, this same mental model carries over: keep the workflow config the same, replace or add a tracing exporter such as LangSmith, Langfuse, OpenTelemetry collector, or Arize AX, and inspect the same span hierarchy in that platform.

## 7. Trace Hygiene

Trace files can contain prompts, model outputs, tool inputs, tool outputs, errors, and timing metadata. Treat them as runtime artifacts: inspect them locally, redact or delete them before sharing, and avoid committing traces that contain secrets, customer data, or private prompts.